In [2]:
import pandas as pd
import numpy as np
from scipy import stats
import os
os.environ["OMP_NUM_THREADS"] = "1"

In [9]:
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from dash import Dash, dcc, html, Input, Output, callback
import dash_bootstrap_components as dbc

In [3]:
states = pd.read_csv("data-oKFfw.csv")
states.head(5)

,SUMLEV,REGION,DIVISION,STATE,NAME,ESTIMATESBASE2020,POPESTIMATE2020,POPESTIMATE2021,POPESTIMATE2022,POPESTIMATE2023,...,DOMESTICMIG2022,DOMESTICMIG2023,DOMESTICMIG2024,DOMESTICMIG2025,RATEDOMESTICMIG2021,RATEDOMESTICMIG2022,RATEDOMESTICMIG2023,RATEDOMESTICMIG2024,RATEDOMESTICMIG2025,rank
0,40,South,South Atlantic,45,South Carolina,5118250,5131992,5194346,5288957,5390798,...,83341,79536,66367,66622,13.2,15.9,14.9,12.2,12.05,1st
1,40,West,Mountain,16,Idaho,1839123,1849328,1904855,1942951,1970497,...,28019,14713,15975,19915,27.9,14.6,7.5,8.0,9.88,2nd
2,40,South,South Atlantic,37,North Carolina,10441392,10450215,10565503,10705768,10871849,...,98454,98929,83059,84064,9.9,9.3,9.2,7.6,7.56,3rd
3,40,South,South Atlantic,10,Delaware,989950,991890,1005130,1020279,1035354,...,12530,9872,8040,6855,13.7,12.4,9.6,7.7,6.50,4th
4,40,South,East South Central,47,Tennessee,6912319,6927736,6966687,7063325,7153029,...,82316,60397,46496,42389,6.7,11.7,8.5,6.5,5.82,5th


In [4]:
states['pop_growth_pct'] = (states['POPESTIMATE2025'] - states['POPESTIMATE2020']) / states['POPESTIMATE2020'] * 100
states['avg_mig_rate']   = states[['RATEDOMESTICMIG2021','RATEDOMESTICMIG2022',
                            'RATEDOMESTICMIG2023','RATEDOMESTICMIG2024',
                            'RATEDOMESTICMIG2025']].mean(axis=1)
states['total_net_mig']  = states[['DOMESTICMIG2021','DOMESTICMIG2022',
                            'DOMESTICMIG2023','DOMESTICMIG2024',
                            'DOMESTICMIG2025']].sum(axis=1)
states['mig_trend']  = states['RATEDOMESTICMIG2025'] - states['RATEDOMESTICMIG2021']

In [8]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
X_c = states[['avg_mig_rate','pop_growth_pct','mig_trend','RATEDOMESTICMIG2025','RATEDOMESTICMIG2021']].fillna(0)
sc  = StandardScaler()
km  = KMeans(n_clusters=4, random_state=42, n_init=10)
states['cluster'] = km.fit_predict(sc.fit_transform(X_c))
CLUSTER_MAP = {0:'Migration Magnet', 1:'Declining Hub', 2:'Stable State', 3:'Emerging Gainer'}
# Remap by mean rate descending
order = states.groupby('cluster')['RATEDOMESTICMIG2025'].mean().sort_values(ascending=False).index.tolist()
remap = {old: ['Migration Magnet','Emerging Gainer','Stable State','Declining Hub'][i]
         for i, old in enumerate(order)}
states['cluster_label'] = states['cluster'].map(remap)